In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, 
    recall_score, f1_score, matthews_corrcoef
)

# Load dataset from parent directory or local copy
df = pd.read_excel('../UCI_Credit_Card.xlsx', header=0)
df.columns = [str(c).strip() for c in df.columns]

target_cols = [c for c in df.columns if 'default' in c.lower()]
df.rename(columns={target_cols[0]: 'default'}, inplace=True)

if 'ID' in df.columns:
    df.drop(columns=['ID'], inplace=True)

X = df.drop(columns=['default'])
y = df['default']

imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

def get_trained_pipelines(X_train, y_train):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=5000, random_state=42),
        "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
        "kNN": KNeighborsClassifier(n_neighbors=5),
        "Naive Bayes": GaussianNB(),
        "Random Forest (Ensemble)": RandomForestClassifier(
            n_estimators=200, max_depth=8, random_state=42
        ),
    }

    fitted_pipelines = {}
    for name, estimator in models.items():
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", estimator)
        ])
        pipe.fit(X_train, y_train)
        fitted_pipelines[name] = pipe

    return fitted_pipelines

fitted_pipelines = get_trained_pipelines(X_train, y_train)

for name, pipe in fitted_pipelines.items():
    file_name = name.lower().replace(" (ensemble)", "_ensemble").replace(" ", "_") + ".pkl"
    joblib.dump(pipe, file_name)